**GA with Encoders**

**Trainning AE-GA with RF**

In [2]:
#training GA with RF (AE + GA)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score


# 1. Data Loading & Preparation

df = pd.read_excel('../minmax.xlsx') 
data = df.values
labels = pd.read_csv('../idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model

class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        #Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh() 
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance losses
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32


# Initialize model, optimizer, and scheduler
model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train).numpy()
    test_latent = model.encoder(X_test).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

from deap import base, creator, tools, algorithms
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import random
import warnings
warnings.filterwarnings("ignore")

print("Starting GA")
print("latent data:", train_latent.shape)

# Define all the needed parameters

LATENT_FEATURES = train_latent.shape[1]
N_GENERATIONS = 20
POPULATION_SIZE = 100
P_CROSSOVER = 0.7
P_MUTATION = 0.2
TOURNAMENT_SIZE = 3
N_SELECTED_LATENT_FEATURES = 10

try:
    del creator.FitnessMax
    del creator.Individual
except:
    pass

# Create types for GA
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=LATENT_FEATURES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate_individual_with_latent(individual):
    # Get selected latent features
    selected_indices = [i for i, val in enumerate(individual) if val == 1]
    
    # If too few features are selected, penalize the individual
    if len(selected_indices) < 2:
        return 0.0,
    
    # Create a modified input tensor with zeros for non-selected features
    modified_latent = train_latent[:, selected_indices]
    
    # Train a simple classifier on the selected latent features
    classifier = RandomForestClassifier(random_state=42)
    scores = cross_val_score(classifier, modified_latent, y_train_np, cv=5)
    accuracy = scores.mean()

    return accuracy,

toolbox.register("evaluate", evaluate_individual_with_latent)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

def run_ga():
    # Initialize population
    population = toolbox.population(n=POPULATION_SIZE)
    
    # Statistics to track progress
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # Hall of fame to keep track of best individuals
    hof = tools.HallOfFame(1)
    
    # Running the algorithm
    population, logbook = algorithms.eaSimple(
        population, 
        toolbox, 
        cxpb=P_CROSSOVER, 
        mutpb=P_MUTATION, 
        ngen=N_GENERATIONS, 
        stats=stats, 
        halloffame=hof, 
        verbose=True
    )
    
    return population, logbook, hof


population, logbook, hof = run_ga()

# Get the best individual
best_individual = hof[0]
selected_latent_features = [i for i, val in enumerate(best_individual) if val == 1]
num_selected = len(selected_latent_features)

print(f"Number of selected latent features: {num_selected} out of {LATENT_FEATURES}")
print('The selected features: ', selected_latent_features)
print(f"Best fitness: {best_individual.fitness.values[0]}")


#testing the ga selected features on the models
print('Testing GA Selected Features: ')
modified_latent_train = train_latent[:, selected_latent_features]
modified_latent_test = test_latent[:, selected_latent_features]


# --- SVM Classifier ---
svm_model = SVC(random_state=42)
svm_model.fit(modified_latent_train, y_train_np)
svm_preds = svm_model.predict(modified_latent_test)

svm_acc = accuracy_score(y_test_np, svm_preds)
svm_percision = precision_score(y_test_np, svm_preds, average='macro')
svm_recall = recall_score(y_test_np, svm_preds, average='macro')
svm_f1 = f1_score(y_test_np, svm_preds, average='macro')

print(f"SVM Accuracy on latent features selected by GA: {svm_acc * 100:.4f}")
print(f"SVM Percision on latent features selected by GA: {svm_percision * 100:.4f}")
print(f"SVM Recall on latent features selected by GA: {svm_recall * 100:.4f}")
print(f"SVM F1-Score on latent features selected by GA: {svm_f1 * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(modified_latent_train, y_train_np)
nbc_preds = nbc_model.predict(modified_latent_test)

nbc_acc = accuracy_score(y_test_np, nbc_preds)
nbc_percision = precision_score(y_test_np, nbc_preds, average='macro')
nbc_recall = recall_score(y_test_np, nbc_preds, average='macro')
nbc_f1 = f1_score(y_test_np, nbc_preds, average='macro')

print(f"Naïve Bayes Accuracy on latent features selected by GA: {nbc_acc * 100:.4f}")
print(f"Naïve Bayes Percision on latent features selected by GA: {nbc_percision * 100:.4f}")
print(f"Naïve Bayes Recall on latent features selected by GA: {nbc_recall * 100:.4f}")
print(f"Naïve Bayes F1-Score on latent features selected by GA: {nbc_f1 * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(modified_latent_train, y_train_np)
rf_preds = rf_model.predict(modified_latent_test)

rf_acc = accuracy_score(y_test_np, rf_preds)
rf_percision = precision_score(y_test_np, rf_preds, average='macro')
rf_recall = recall_score(y_test_np, rf_preds, average='macro')
rf_f1 = f1_score(y_test_np, rf_preds, average='macro')

print(f"Random Forest Accuracy on latent features selected by GA: {rf_acc * 100:.4f}")
print(f"Random Forest Percision on latent features selected by GA: {rf_percision * 100:.4f}")
print(f"Random Forest Recall on latent features selected by GA: {rf_recall * 100:.4f}")
print(f"Random Forest F1-Score on latent features selected by GA: {rf_f1 * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.1465
Epoch [2/50], Loss: 0.7878
Epoch [3/50], Loss: 0.6688
Epoch [4/50], Loss: 0.5994
Epoch [5/50], Loss: 0.5437
Epoch [6/50], Loss: 0.5187
Epoch [7/50], Loss: 0.4274
Epoch [8/50], Loss: 0.4508
Epoch [9/50], Loss: 0.4238
Epoch [10/50], Loss: 0.4322
Epoch [11/50], Loss: 0.4676
Epoch [12/50], Loss: 0.4144
Epoch [13/50], Loss: 0.3951
Epoch [14/50], Loss: 0.4304
Epoch [15/50], Loss: 0.4548
Epoch [16/50], Loss: 0.4300
Epoch [17/50], Loss: 0.3536
Epoch [18/50], Loss: 0.3667
Epoch [19/50], Loss: 0.4498
Epoch [20/50], Loss: 0.3786
Epoch [21/50], Loss: 0.4513
Epoch [22/50], Loss: 0.3986
Epoch [23/50], Loss: 0.4278
Epoch [24/50], Loss: 0.4417
Epoch [25/50], Loss: 0.3799
Epoch [26/50], Loss: 0.3850
Epoch [27/50], Loss: 0.3782
Epoch [28/50], Loss: 0.3896
Epoch [29/50], Loss: 0.3823
Epoch [30/50], Loss: 0.3854
Epoch [31/50], Loss: 0.4335
Epoch [32/50], Loss: 0.3687
Epoch [33/50], Loss: 0.3642
Epoch [34/50], Loss: 0.3703
Epoch [35/50], Loss: 0.3440


**Training VAE-GA with RF**

In [ ]:
# VAE + GA

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score


# 1. Data Loading & Preparation

df = pd.read_excel('minmax_normalized_dataset.xlsx') 
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model

class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        #Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh() 
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance losses
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32


# Initialize model, optimizer, and scheduler
model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train).numpy()
    test_latent = model.encoder(X_test).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

from deap import base, creator, tools, algorithms
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import random
import warnings
warnings.filterwarnings("ignore")

print("Starting GA")
print("latent data:", train_latent.shape)

# Define all the needed parameters

LATENT_FEATURES = train_latent.shape[1]
N_GENERATIONS = 20
POPULATION_SIZE = 100
P_CROSSOVER = 0.7
P_MUTATION = 0.2
TOURNAMENT_SIZE = 3
N_SELECTED_LATENT_FEATURES = 10

try:
    del creator.FitnessMax
    del creator.Individual
except:
    pass

# Create types for GA
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=LATENT_FEATURES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate_individual_with_latent(individual):
    # Get selected latent features
    selected_indices = [i for i, val in enumerate(individual) if val == 1]
    
    # If too few features are selected, penalize the individual
    if len(selected_indices) < 2:
        return 0.0,
    
    # Create a modified input tensor with zeros for non-selected features
    modified_latent = train_latent[:, selected_indices]
    
    # Train a simple classifier on the selected latent features
    classifier = RandomForestClassifier(random_state=42)
    scores = cross_val_score(classifier, modified_latent, y_train_np, cv=5)
    accuracy = scores.mean()

    return accuracy,

toolbox.register("evaluate", evaluate_individual_with_latent)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

def run_ga():
    # Initialize population
    population = toolbox.population(n=POPULATION_SIZE)
    
    # Statistics to track progress
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # Hall of fame to keep track of best individuals
    hof = tools.HallOfFame(1)
    
    # Running the algorithm
    population, logbook = algorithms.eaSimple(
        population, 
        toolbox, 
        cxpb=P_CROSSOVER, 
        mutpb=P_MUTATION, 
        ngen=N_GENERATIONS, 
        stats=stats, 
        halloffame=hof, 
        verbose=True
    )
    
    return population, logbook, hof


population, logbook, hof = run_ga()

# Get the best individual
best_individual = hof[0]
selected_latent_features = [i for i, val in enumerate(best_individual) if val == 1]
num_selected = len(selected_latent_features)

print(f"Number of selected latent features: {num_selected} out of {LATENT_FEATURES}")
print('The selected features: ', selected_latent_features)
print(f"Best fitness: {best_individual.fitness.values[0]}")


#testing the ga selected features on the models
print('Testing GA Selected Features: ')
modified_latent_train = train_latent[:, selected_latent_features]
modified_latent_test = test_latent[:, selected_latent_features]


# --- SVM Classifier ---
svm_model = SVC(random_state=42)
svm_model.fit(modified_latent_train, y_train_np)
svm_preds = svm_model.predict(modified_latent_test)

svm_acc = accuracy_score(y_test_np, svm_preds)
svm_percision = precision_score(y_test_np, svm_preds, average='macro')
svm_recall = recall_score(y_test_np, svm_preds, average='macro')
svm_f1 = f1_score(y_test_np, svm_preds, average='macro')

print(f"SVM Accuracy on latent features selected by GA: {svm_acc * 100:.4f}")
print(f"SVM Percision on latent features selected by GA: {svm_percision * 100:.4f}")
print(f"SVM Recall on latent features selected by GA: {svm_recall * 100:.4f}")
print(f"SVM F1-Score on latent features selected by GA: {svm_f1 * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(modified_latent_train, y_train_np)
nbc_preds = nbc_model.predict(modified_latent_test)

nbc_acc = accuracy_score(y_test_np, nbc_preds)
nbc_percision = precision_score(y_test_np, nbc_preds, average='macro')
nbc_recall = recall_score(y_test_np, nbc_preds, average='macro')
nbc_f1 = f1_score(y_test_np, nbc_preds, average='macro')

print(f"Naïve Bayes Accuracy on latent features selected by GA: {nbc_acc * 100:.4f}")
print(f"Naïve Bayes Percision on latent features selected by GA: {nbc_percision * 100:.4f}")
print(f"Naïve Bayes Recall on latent features selected by GA: {nbc_recall * 100:.4f}")
print(f"Naïve Bayes F1-Score on latent features selected by GA: {nbc_f1 * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(modified_latent_train, y_train_np)
rf_preds = rf_model.predict(modified_latent_test)

rf_acc = accuracy_score(y_test_np, rf_preds)
rf_percision = precision_score(y_test_np, rf_preds, average='macro')
rf_recall = recall_score(y_test_np, rf_preds, average='macro')
rf_f1 = f1_score(y_test_np, rf_preds, average='macro')

print(f"Random Forest Accuracy on latent features selected by GA: {rf_acc * 100:.4f}")
print(f"Random Forest Percision on latent features selected by GA: {rf_percision * 100:.4f}")
print(f"Random Forest Recall on latent features selected by GA: {rf_recall * 100:.4f}")
print(f"Random Forest F1-Score on latent features selected by GA: {rf_f1 * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/20], Loss: 1.3370
Epoch [2/20], Loss: 0.9781
Epoch [3/20], Loss: 0.7969
Epoch [4/20], Loss: 0.6912
Epoch [5/20], Loss: 0.6107
Epoch [6/20], Loss: 0.5256
Epoch [7/20], Loss: 0.5200
Epoch [8/20], Loss: 0.4308
Epoch [9/20], Loss: 0.4721
Epoch [10/20], Loss: 0.4536
Epoch [11/20], Loss: 0.3795
Epoch [12/20], Loss: 0.4454
Epoch [13/20], Loss: 0.3811
Epoch [14/20], Loss: 0.4285
Epoch [15/20], Loss: 0.4433
Epoch [16/20], Loss: 0.3833
Epoch [17/20], Loss: 0.4063
Epoch [18/20], Loss: 0.4064
Epoch [19/20], Loss: 0.3608
Epoch [20/20], Loss: 0.3956
Starting GA
latent data: (354, 32)
gen	nevals	avg     	std       	min     	max
0  	100   	0.999293	0.00161787	0.991509	1  
1  	80    	0.999774	0.000863025	0.994366	1  
2  	81    	0.999435	0.0014411  	0.991549	1  
3  	69    	0.999633	0.00123955 	0.991549	1  
4  	67    	0.999803	0.000821744	0.994366	1  
5  	71    	0.999915	0.000482827	0.997143	1  
6  	81    	0.999577	0.00128392 	0.994366	1  
7  	71    	0.999859	0.00073

KeyboardInterrupt: 

**Training PSO with RF**

In [6]:
#PSO WITH RF

import pyswarms as ps
from pyswarms.discrete import BinaryPSO
from pyswarms.utils.search import RandomSearch
from pyswarms.single import GlobalBestPSO
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

X_df = pd.read_excel('minmax_normalized_dataset.xlsx')
y_df = pd.read_csv('idC_with_header.csv')

X = X_df.to_numpy()
y = y_df.to_numpy()

options = {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
pso = BinaryPSO(n_particles=30, dimensions=X.shape[1], options=options)

import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
 
def fitness(particles):
    n_particles = particles.shape[0]
    scores = np.zeros(n_particles)
    alpha = 0.99
    for i, particle in enumerate(particles):
        mask = particle.astype(bool)
        if not mask.any():
            scores[i] = 1.0
            continue
        X_sub = X[:, mask]
        acc = cross_val_score(RandomForestClassifier(random_state=42),
                              X_sub, y,
                              cv=5,
                              scoring='accuracy',
                              n_jobs=-1).mean()
        sparsity = mask.sum() / X.shape[1]
        scores[i] = alpha * (1 - acc) + (1 - alpha) * sparsity
    return scores

best_cost, best_pos = pso.optimize(fitness, iters=100)
print("Best Cost: ", best_cost, "\nBest POS: ", best_pos)

selected = np.where(best_pos == 1)[0]
X_selected = X[:, selected]

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

svm_model = SVC(random_state=42)
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)

svm_acc = accuracy_score(y_test, svm_preds)
svm_percision = precision_score(y_test, svm_preds, average='macro')
svm_recall = recall_score(y_test, svm_preds, average='macro')
svm_f1 = f1_score(y_test, svm_preds, average='macro')

print(f"SVM Accuracy on features selected by PSO: {svm_acc * 100:.4f}")
print(f"SVM Percision on features selected by PSO: {svm_percision * 100:.4f}")
print(f"SVM Recall on features selected by PSO: {svm_recall * 100:.4f}")
print(f"SVM F1-Score on features selected by PSO: {svm_f1 * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(X_train, y_train)
nbc_preds = nbc_model.predict(X_test)

nbc_acc = accuracy_score(y_test, nbc_preds)
nbc_percision = precision_score(y_test, nbc_preds, average='macro')
nbc_recall = recall_score(y_test, nbc_preds, average='macro')
nbc_f1 = f1_score(y_test, nbc_preds, average='macro')

print(f"Naïve Bayes Accuracy on features selected by PSO: {nbc_acc * 100:.4f}")
print(f"Naïve Bayes Percision on features selected by PSO: {nbc_percision * 100:.4f}")
print(f"Naïve Bayes Recall on features selected by PSO: {nbc_recall * 100:.4f}")
print(f"Naïve Bayes F1-Score on features selected by PSO: {nbc_f1 * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, rf_preds)
rf_percision = precision_score(y_test, rf_preds, average='macro')
rf_recall = recall_score(y_test, rf_preds, average='macro')
rf_f1 = f1_score(y_test, rf_preds, average='macro')

print(f"Random Forest Accuracy on features selected by PSO: {rf_acc * 100:.4f}")
print(f"Random Forest Percision on features selected by PSO: {rf_percision * 100:.4f}")
print(f"Random Forest Recall on features selected by PSO: {rf_recall * 100:.4f}")
print(f"Random Forest F1-Score on features selected by PSO: {rf_f1 * 100:.4f}")


2025-05-06 22:23:44,432 - pyswarms.discrete.binary - INFO - Optimize for 100 iters with {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
pyswarms.discrete.binary: 100%|██████████|100/100, best_cost=0.0607
2025-05-06 23:04:22,284 - pyswarms.discrete.binary - INFO - Optimization finished | best cost: 0.060748564294631664, best pos: [1 1 1 0 0 1 0 0 1 0 1 0 1 1 0 0 0 0 1 1 0 0 1 1 0 0 0 0 0 0 0 0 1 1 1 0 1
 0 1 0 1 0 1 1 0 0 0 1 1 1 1 1 0 1 0 1 1 0 0 0 1 0 1 1 0 1 1 1 0 0 1 1 0 1
 1 0 1 1 0 0 1 1 1 1 1 1 1 0 1 0 1 0 0 0 0 0 1 1 0 0 1 0 1 0 0 0 0 0 1 0 1
 1 0 1 1 0 0 1 0 0 1 0 0 1 1 0 1 0 0 0 0 1 1 1 1 0 1 0 0 1 0 0 1 0 0 1 1 1
 1 1 0 0 0 0 1 1 1 0 1 1 1 0 0 1 0 0 1 1 1 0 1 1 1 0 1 0 0 0 1 1 0 0 1 1 1
 1 0 0 1 0 1 0 1 1 0 1 1 0 0 0 0 1 1 0 1 0 0 1 0 0 1 1 0 1 0 0 1 0 0 0 0 1
 0 0 0 0 1 1 0 1 1 1 0 1 0 0 1 0 0 1 0 0 0 1 0 0 1 0 0 0 0 0 1 0 1 0 0 1 0
 1 0 1 1 0 0 0 1 0 0 0 1 1 0 1 1 1 1 0 0 1 0 1 0 0 0 1 0 0 1 0 0 0 0 1 0 1
 1 0 0 0 1 1 0 1 1 0 1 0 0 0 0 0 0 1 0 0 1 0 0 0 1 1 0 0 1 0 0 1 1 1 1 1

Best Cost:  0.060748564294631664 
Best POS:  [1 1 1 0 0 1 0 0 1 0 1 0 1 1 0 0 0 0 1 1 0 0 1 1 0 0 0 0 0 0 0 0 1 1 1 0 1
 0 1 0 1 0 1 1 0 0 0 1 1 1 1 1 0 1 0 1 1 0 0 0 1 0 1 1 0 1 1 1 0 0 1 1 0 1
 1 0 1 1 0 0 1 1 1 1 1 1 1 0 1 0 1 0 0 0 0 0 1 1 0 0 1 0 1 0 0 0 0 0 1 0 1
 1 0 1 1 0 0 1 0 0 1 0 0 1 1 0 1 0 0 0 0 1 1 1 1 0 1 0 0 1 0 0 1 0 0 1 1 1
 1 1 0 0 0 0 1 1 1 0 1 1 1 0 0 1 0 0 1 1 1 0 1 1 1 0 1 0 0 0 1 1 0 0 1 1 1
 1 0 0 1 0 1 0 1 1 0 1 1 0 0 0 0 1 1 0 1 0 0 1 0 0 1 1 0 1 0 0 1 0 0 0 0 1
 0 0 0 0 1 1 0 1 1 1 0 1 0 0 1 0 0 1 0 0 0 1 0 0 1 0 0 0 0 0 1 0 1 0 0 1 0
 1 0 1 1 0 0 0 1 0 0 0 1 1 0 1 1 1 1 0 0 1 0 1 0 0 0 1 0 0 1 0 0 0 0 1 0 1
 1 0 0 0 1 1 0 1 1 0 1 0 0 0 0 0 0 1 0 0 1 0 0 0 1 1 0 0 1 0 0 1 1 1 1 1 0
 1 1 1 1 0 1 0 0 1 0 1 1 1 0 0 1 1 0 0 1 0 1 1 0 0 1 1 1 0 1 0 0 1 0 1 1 0
 0 1 1 1 1 0 0 1 1 0 1 1 1 1 1 1 0 0 1 0 1 1 0 1 0 0 1 1 0 1 1 0 1 1 0 1 1
 1 1 0 1 0 1 1 1 0 1 0 0 0 0 1 1 0 1 1 0 0 1 1 0 0 0 1 1 1 0 1 0 1 1 1 0 1
 0 1 1 0 0 1 0 1 0 0 1 0 1 1 1 0 0 1 1 0 0 0 1 0 0 1 0 